In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
import numpy as np
from pathlib import Path

from torch.utils.data import Dataset, DataLoader, TensorDataset, ConcatDataset

In [12]:
# Check PyTorch version and CUDA support
import sys
print(f"Python executable: {sys.executable}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_built():
    print(f"MPS available: {torch.backends.mps.is_available()}")

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_built() and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"\nUsing {device} device")

Python executable: c:\Users\alexa\AppData\Local\Programs\Python\Python313\python.exe
PyTorch version: 2.10.0+cu130
CUDA available: True
CUDA version: 13.0
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 3060 Ti

Using cuda device


In [13]:
PREDICTIONS_H = 20
INPUT_LENGTH = 105
LEARNING_RATE = 5e-4

In [14]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=1, bidirectional=True, dropout=0.2):
        super().__init__()

        self.data_in = nn.Linear(input_dim, emb_dim)
        self.dropout_in = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers,
                            bidirectional=bidirectional, batch_first=True, 
                            dropout=dropout if num_layers > 1 else 0)
        self.bidirectional = bidirectional
        self.hidden_dim = hidden_dim

    def forward(self, x):
        emb = self.data_in(x)
        emb = self.dropout_in(emb)
        outputs, (h_n, c_n) = self.lstm(emb)
        return outputs, (h_n, c_n)

class AttentionModule(nn.Module):
    def __init__(self, encoding_dim, decoding_dim, attention_dim):
        super().__init__()
        self.enc = nn.Linear(encoding_dim, attention_dim, bias=False)
        self.dec = nn.Linear(decoding_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)
        self._warned = False

    def forward(self, dec_hidden, encoding_outputs, mask=None):
        if encoding_outputs.dim() == 2:
            encoding_outputs = encoding_outputs.unsqueeze(0)
        if dec_hidden.dim() == 1:
            dec_hidden = dec_hidden.unsqueeze(0)

        enc_proj = self.enc(encoding_outputs)
        dec_proj = self.dec(dec_hidden).unsqueeze(1) 

        scores = self.v(torch.tanh(enc_proj + dec_proj)).squeeze(-1)

        if mask is not None:
            if mask.dim() == 1:
                mask = mask.unsqueeze(0)
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = F.softmax(scores, dim=1)

        if weights.dim() == 1:
            weights = weights.unsqueeze(0)
        weights_bmm = weights.unsqueeze(1) if weights.dim() == 2 else weights

        if encoding_outputs.dim() != 3:
            encoding_outputs = encoding_outputs.unsqueeze(0)

        if weights_bmm.dim() != 3 or encoding_outputs.dim() != 3:
            if not self._warned:
                print("AttentionModule error: shapes before bmm:",
                      "weights_bmm.shape=", getattr(weights_bmm, "shape", None),
                      "encoding_outputs.shape=", getattr(encoding_outputs, "shape", None))
                self._warned = True
            raise RuntimeError(f"AttentionModule: expected 3D tensors for bmm but got "
                               f"{weights_bmm.dim()}D and {encoding_outputs.dim()}D tensors.")
        context = torch.bmm(weights_bmm, encoding_outputs).squeeze(1)

        return context, weights

class Decoder(nn.Module):
    def __init__(self, encoding_dim, decoding_dim, attention_dim, out_dim=1, emb_dim=32, dropout=0.2):
        super().__init__()

        self.attnMod = AttentionModule(encoding_dim, decoding_dim, attention_dim) 
        self.lstmCell = nn.LSTMCell(encoding_dim + emb_dim, decoding_dim)
        self.input_proj = nn.Linear(1, emb_dim)
        self.dropout = nn.Dropout(dropout)
        # Combine decoder hidden state and context more effectively
        self.out = nn.Linear(decoding_dim + encoding_dim, out_dim)
        # Add a layer to help with sequence learning
        self.hidden_proj = nn.Linear(decoding_dim, decoding_dim)

    def forward(self, encoding_outputs, dec_h, dec_c, iters, first_input=None, mask=None, teacher_forcing=None, teacher_forcing_prob=0.0):
        batch = encoding_outputs.size(0)
        device = encoding_outputs.device
        predictions = []
        input_t = first_input if first_input is not None else torch.zeros(batch, 1, device=device)
        
        # Use teacher forcing during training if provided
        use_teacher_forcing = teacher_forcing is not None and self.training and torch.rand(1).item() < teacher_forcing_prob

        for i in range(iters):
            emb_in = self.input_proj(input_t)
            # Only apply dropout during training
            emb_in = self.dropout(emb_in) if self.training else emb_in
            # Get attention context - this should vary based on decoder hidden state
            context, weights = self.attnMod(dec_h, encoding_outputs, mask)
            lstm_in = torch.cat([emb_in, context], dim=1)
            dec_h, dec_c = self.lstmCell(lstm_in, (dec_h, dec_c))
            # Project hidden state to help with variation
            dec_h_proj = torch.tanh(self.hidden_proj(dec_h))
            # Only apply dropout during training
            dec_h_dropped = self.dropout(dec_h_proj) if self.training else dec_h_proj
            # Combine hidden state and context for output
            out = self.out(torch.cat([dec_h_dropped, context], dim=1))
            predictions.append(out)
            
            # Use teacher forcing or predicted value
            if use_teacher_forcing and i < teacher_forcing.size(1):
                input_t = teacher_forcing[:, i:i+1]  # Use actual target
            else:
                # During training, keep gradient flow. During inference, detach to prevent issues
                if self.training:
                    input_t = out  # Keep gradient flow during training
                else:
                    input_t = out.detach()  # Detach during inference to prevent accumulation

        return predictions

class Wrapper(nn.Module):
    def __init__(self, input_dim, enc_emb, enc_hid, dec_hid, attn_dim, bidir=True, dropout=0.2):
        super().__init__()

        self.encoder = Encoder(input_dim, enc_emb, enc_hid, bidirectional=bidir, dropout=dropout)
        self.enc_dim = enc_hid * (2 if bidir else 1)
        self.h_proj = nn.Linear(self.enc_dim * 1, dec_hid)
        self.c_proj = nn.Linear(self.enc_dim * 1, dec_hid)
        self.decoder = Decoder(self.enc_dim, dec_hid, attn_dim, out_dim=1, dropout=dropout)
        self.H = PREDICTIONS_H

    def forward(self, x, first_input=None, teacher_forcing=None, teacher_forcing_prob=0.0):
        enc_out, (h_n, c_n) = self.encoder(x)
        batch = x.size(0)
        h_flat = h_n.permute(1, 0, 2).contiguous().view(batch, -1)
        c_flat = c_n.permute(1, 0, 2).contiguous().view(batch, -1)
        dec_h = torch.tanh(self.h_proj(h_flat))
        dec_c = torch.tanh(self.c_proj(c_flat))
        
        # Use the last price from input as first decoder input if not provided
        if first_input is None:
            # Extract the last price value from the input (price is first feature, index 0)
            first_input = x[:, -1, 0:1]  # Last timestep, price feature
        
        preds = self.decoder(enc_out, dec_h, dec_c, self.H, first_input=first_input, 
                             teacher_forcing=teacher_forcing, teacher_forcing_prob=teacher_forcing_prob)
        return preds

In [15]:
def minmax_scale(arr, eps=1e-8):
    arr = np.asarray(arr, dtype=np.float32)
    mn = arr.min()
    mx = arr.max()
    denom = mx - mn

    if denom < eps: scaled = np.zeros_like(arr, dtype=np.float32)
    else: scaled = (arr - mn) / (denom)

    return scaled.astype(np.float32), float(mn), float(mx)

class StockDataset(Dataset):
    def __init__(self, name, df):
        self.input_len=INPUT_LENGTH
        self.pred_len=PREDICTIONS_H
        self.name = name

        c_prices = df["Close"].to_numpy()
        scaled, mn, mx = minmax_scale(c_prices)
        self.scaler = (mn, mx)
        self.prices = torch.tensor(scaled, dtype=torch.float32)
        self.rsi = torch.tensor(df["Rsi"].to_numpy(), dtype=torch.float32) / 100
        self.k = torch.tensor(df["%K"].to_numpy(), dtype=torch.float32) / 100
        self.d = torch.tensor(df["%D"].to_numpy(), dtype=torch.float32) / 100
        dividends = df["Dividends"].to_numpy() if "Dividends" in df.columns else np.zeros(len(df), dtype=np.float32)
        self.dividends = torch.tensor(dividends, dtype=torch.float32)

        self.features = torch.stack([self.prices, self.rsi, self.k, self.d, self.dividends], dim=1)

    def __len__(self):
        return len(self.features) - self.input_len - self.pred_len
    
    def __repr__(self):
        return self.name + f" of len {self.__len__()}"
    
    def __getitem__(self, idx):
        x = self.features[idx:idx+self.input_len]
        y = self.prices[idx+self.input_len:idx+self.input_len+self.pred_len]

        return x, y


In [16]:
SPLIT_RATIO = 0.7
data_dir = Path("stocks")
files = list(data_dir.glob("*/*/*.parquet")) or list(data_dir.glob("*/*/*.csv"))
train_datasets = []
val_datasets = []

for file in files:
    name = file.stem
    df = pl.read_parquet(file) if file.suffix == ".parquet" else pl.read_csv(file)
    split_idx = int(len(df) * SPLIT_RATIO)

    train_df = df.head(split_idx)
    validation_df = df.tail(len(df) - split_idx)

    train_set = StockDataset(name, train_df)
    val_set = StockDataset(name, validation_df)

    if len(train_set) > 0:
        train_datasets.append(train_set)
    if len(val_set) > 0:
        val_datasets.append(val_set)

In [17]:
train_ds = ConcatDataset(train_datasets) if len(train_datasets) > 0 else None
val_ds = ConcatDataset(val_datasets) if len(val_datasets) > 0 else None

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True) if train_ds is not None else None
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False) if val_ds is not None else None

In [18]:
print(f"Number of training datasets: {len(train_datasets)}")
print(f"Number of validation datasets: {len(val_datasets)}")
if train_ds is not None:
    print(f"Total training samples: {len(train_ds)}")
if val_ds is not None:
    print(f"Total validation samples: {len(val_ds)}")

if train_loader is not None:
    sample_x, sample_y = next(iter(train_loader))
    print(f"\nSample batch shapes:")
    print(f"  X shape: {sample_x.shape}")
    print(f"  Y shape: {sample_y.shape}")


Number of training datasets: 25
Number of validation datasets: 25
Total training samples: 17166
Total validation samples: 5580

Sample batch shapes:
  X shape: torch.Size([32, 105, 5])
  Y shape: torch.Size([32, 20])


In [19]:
def prepare_preds_tensor(preds_list):
    preds_tensor = torch.stack(preds_list, dim=1)
    if preds_tensor.size(-1) == 1:
        preds_tensor = preds_tensor.squeeze(-1)
    return preds_tensor

def prepare_target_tensor(y):
    if y.dim() == 3 and y.size(-1) == 1:
        return y.squeeze(-1)
    return y

def stack_preds(preds_list):
    out = torch.stack(preds_list, dim=1) 
    if out.size(-1) == 1: out = out.squeeze(-1)
    return out

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def train_and_validate(model, train_loader, val_loader,
                        epochs=50, weight_decay=1e-5, clip_grad=1.0, save_path="model.pt", 
                        save_every=10, save_dir="models", patience=7, min_lr=1e-6, teacher_forcing_prob=0.8):
    import os
    
    os.makedirs(save_dir, exist_ok=True)
    
    model.to(device)
    model.apply(init_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=min_lr
    )
    criterion = nn.MSELoss()

    train_losses = []
    val_losses = []
    
    best_val_loss = float("inf")
    patience_counter = 0
    best_epoch = 0
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_n = 0
        # Scheduled sampling: decrease teacher forcing over time
        current_tf_prob = teacher_forcing_prob * (1.0 - (epoch - 1) / epochs * 0.5)  # Decay to 50% of original

        for x, y in train_loader:
            x = x.to(device).float()
            y = y.to(device).float()

            if x.dim() == 2: x = x.unsqueeze(-1)
            if torch.isnan(x).any() or torch.isinf(x).any() or torch.isnan(y).any() or torch.isinf(y).any():
                continue

            optimizer.zero_grad()
            # Prepare teacher forcing target - ensure it's (batch, sequence_length)
            teacher_target = y if y.dim() == 2 else y.squeeze(-1) if y.dim() == 3 else y
            # Use teacher forcing during training with scheduled sampling
            preds_list = model(x, teacher_forcing=teacher_target, teacher_forcing_prob=current_tf_prob)
            preds = stack_preds(preds_list)

            if preds.dim() == 3 and y.dim() == 2: y_t = y.unsqueeze(-1)
            else: y_t = y

            if torch.isnan(preds).any() or torch.isinf(preds).any():
                continue

            loss = criterion(preds, y_t)
            if not torch.isfinite(loss):
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()

            train_loss += loss.item()
            train_n += 1

        avg_train = train_loss / train_n if train_n > 0 else float("nan")

        model.eval()
        val_loss = 0.0
        val_n = 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device).float()
                y = y.to(device).float()

                if x.dim() == 2: x = x.unsqueeze(-1)
                if torch.isnan(x).any() or torch.isinf(x).any() or torch.isnan(y).any() or torch.isinf(y).any():
                    continue

                # No teacher forcing during validation
                preds_list = model(x, teacher_forcing_prob=0.0)
                preds = stack_preds(preds_list)

                if preds.dim() == 3 and y.dim() == 2: y_t = y.unsqueeze(-1)
                else: y_t = y

                if torch.isnan(preds).any() or torch.isinf(preds).any():
                    continue

                loss = criterion(preds, y_t)
                if not torch.isfinite(loss):
                    continue

                val_loss += loss.item()
                val_n += 1

        avg_val = val_loss / val_n if val_n > 0 else float("nan")
        rmse = avg_val**0.5 if avg_val == avg_val and avg_val != float("inf") else float("nan")
        
        # Update learning rate based on validation loss
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val)
        current_lr = optimizer.param_groups[0]['lr']
        if current_lr < old_lr:
            print(f"  → Learning rate reduced from {old_lr:.2e} to {current_lr:.2e}")

        train_losses.append(avg_train)
        val_losses.append(avg_val)

        # Early stopping logic
        if avg_val == avg_val and avg_val < best_val_loss:
            best_val_loss = avg_val
            best_epoch = epoch
            patience_counter = 0
            print(f"✓ Saving a new best model at epoch {epoch} (Val MSE: {avg_val:.6f})")
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n⏹ Early stopping triggered at epoch {epoch}")
                print(f"   Best validation loss: {best_val_loss:.6f} at epoch {best_epoch}")
                print(f"   No improvement for {patience} epochs")
                break

        if epoch % save_every == 0:
            epoch_save_path = os.path.join(save_dir, f"model_epoch_{epoch}.pt")
            torch.save(model.state_dict(), epoch_save_path)
            print(f"  → Saved checkpoint to {epoch_save_path}")

        print(f"Epoch {epoch:3d} | Train MSE: {avg_train:.6f} | Val MSE: {avg_val:.6f} | Val RMSE: {rmse:.6f} | LR: {current_lr:.2e} | Patience: {patience_counter}/{patience}")

    print(f"\nFinished. Best val MSE: {best_val_loss:.6f} at epoch {best_epoch} (saved to {save_path})")
    return save_path, train_losses, val_losses

In [20]:
model = Wrapper(input_dim=5, enc_emb=32, enc_hid=64, dec_hid=64, attn_dim=32, bidir=True, dropout=0.2)
save_path = "best_wrapper.pt"

save_path_used, train_history, val_history = train_and_validate(
    model,
    train_loader,
    val_loader,
    epochs=50,
    weight_decay=1e-5,  # Add weight decay for regularization
    clip_grad=1.0,
    patience=7,  # Stop if no improvement for 7 epochs
    min_lr=1e-6,  # Minimum learning rate
    teacher_forcing_prob=0.8  # Start with 80% teacher forcing, decay over time
)

✓ Saving a new best model at epoch 1 (Val MSE: 0.017926)
Epoch   1 | Train MSE: 0.010274 | Val MSE: 0.017926 | Val RMSE: 0.133887 | LR: 5.00e-04 | Patience: 0/7
✓ Saving a new best model at epoch 2 (Val MSE: 0.014997)
Epoch   2 | Train MSE: 0.005792 | Val MSE: 0.014997 | Val RMSE: 0.122464 | LR: 5.00e-04 | Patience: 0/7
Epoch   3 | Train MSE: 0.005318 | Val MSE: 0.017454 | Val RMSE: 0.132115 | LR: 5.00e-04 | Patience: 1/7
Epoch   4 | Train MSE: 0.004777 | Val MSE: 0.019628 | Val RMSE: 0.140100 | LR: 5.00e-04 | Patience: 2/7
Epoch   5 | Train MSE: 0.004488 | Val MSE: 0.015457 | Val RMSE: 0.124328 | LR: 5.00e-04 | Patience: 3/7
✓ Saving a new best model at epoch 6 (Val MSE: 0.014652)
Epoch   6 | Train MSE: 0.005136 | Val MSE: 0.014652 | Val RMSE: 0.121046 | LR: 5.00e-04 | Patience: 0/7
Epoch   7 | Train MSE: 0.004618 | Val MSE: 0.022993 | Val RMSE: 0.151634 | LR: 5.00e-04 | Patience: 1/7
✓ Saving a new best model at epoch 8 (Val MSE: 0.014587)
Epoch   8 | Train MSE: 0.004799 | Val MSE: 0

In [21]:
import altair as alt

history_df = pl.DataFrame({
    'Epoch': list(range(1, len(train_history) + 1)) * 2,
    'Loss': train_history + val_history,
    'Type': ['Train'] * len(train_history) + ['Validation'] * len(val_history)
})

chart = alt.Chart(history_df.to_pandas()).mark_line(point=True).encode(
    x=alt.X('Epoch:Q', title='Epoch'),
    y=alt.Y('Loss:Q', title='Mean Squared Error (MSE)'),
    color=alt.Color('Type:N', legend=alt.Legend(title='Type'))
).properties(
    width=600,
    height=300,
    title='Training and Validation Loss Over Time'
).configure_axis(
    gridOpacity=0.3
)

chart.save('training_history.png', scale_factor=2)
chart


alt.Chart(...)

In [22]:
import altair as alt

# Load the best model weights
model.load_state_dict(torch.load(save_path_used if 'save_path_used' in locals() else "model.pt"))
model.eval()
with torch.no_grad():
    x_sample, y_sample = next(iter(val_loader))
    x_sample = x_sample.to(device).float()
    
    # Make predictions without teacher forcing
    preds_list = model(x_sample, teacher_forcing_prob=0.0)
    preds = stack_preds(preds_list)
    
    preds_cpu = preds[:20].cpu().numpy()
    y_cpu = y_sample[:20].cpu().numpy()
    
    time_steps = np.arange(PREDICTIONS_H)
    
    plot_rows = []
    for i in range(min(6, len(preds_cpu))):
        plot_rows.append(pl.DataFrame({
            'Time Step': list(time_steps) * 2,
            'Price': list(y_cpu[i]) + list(preds_cpu[i]),
            'Type': ['Actual'] * PREDICTIONS_H + ['Predicted'] * PREDICTIONS_H,
            'Sample': [f'Sample {i+1}'] * (PREDICTIONS_H * 2)
        }))
    df_plot = pl.concat(plot_rows).to_pandas()
    
    chart = alt.Chart(df_plot).mark_line(point=True).encode(
        x=alt.X('Time Step:Q', title='Prediction Horizon'),
        y=alt.Y('Price:Q', title='Normalized Price'),
        color=alt.Color('Type:N', legend=alt.Legend(title='Type')),
        column=alt.Column('Sample:N', header=alt.Header(title='Actual vs Predicted Stock Prices (Validation Set)'))
    ).properties(
        width=200,
        height=150
    ).configure_axis(
        gridOpacity=0.3
    )
    
    chart.save('predictions.png', scale_factor=2)
    chart
    
    mae = np.mean(np.abs(preds_cpu - y_cpu))
    mse = np.mean((preds_cpu - y_cpu) ** 2)
    rmse = np.sqrt(mse)
    
    print(f"\n{'='*60}")
    print(f"MODEL PERFORMANCE METRICS")
    print(f"{'='*60}")
    print(f"Mean Absolute Error (MAE):  {mae:.6f}")
    print(f"Mean Squared Error (MSE):   {mse:.6f}")
    print(f"Root Mean Squared Error:    {rmse:.6f}")
    print(f"{'='*60}")



MODEL PERFORMANCE METRICS
Mean Absolute Error (MAE):  0.082277
Mean Squared Error (MSE):   0.010070
Root Mean Squared Error:    0.100349
